# Neuron masks from pseudo-FDT — two pipelines

Two independent pipelines run from the same pseudo-FDT field:

- **Pipeline A — SOMA-only** (Section 3). Round-blob detection: high
  FDT-percentile threshold + solidity filter + shape regularisation.
  Output: a clean mask of cell bodies, no dendritic lines.
- **Pipeline B — FULL-NEURON skeleton** (Section 4). Soma carving +
  skeletonisation of the dendrite support + leaf-branch pruning. Output:
  somas (filled) ∪ dilated dendrite trace.

Each pipeline has its own `CONFIG`, override cell, calibration cell
(one image), and batch cell (N images). The FDT computation is shared
in Section 2.


## 1. Imports + repo root


In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

NB_DIR = Path.cwd().resolve()
REPO_ROOT = NB_DIR
while REPO_ROOT.parent != REPO_ROOT and not (REPO_ROOT / ".git").is_dir():
    REPO_ROOT = REPO_ROOT.parent
SRC_ROOT = REPO_ROOT / "src"
for p in (SRC_ROOT, REPO_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from synaptic_ssl.pseudolabels.soma_fdt import (
    derive_size_params,
    resolve_cfg,
    run_soma_on_image,
    visualise_soma_mask,
)

plt.rcParams["figure.dpi"] = 120
print("REPO_ROOT:", REPO_ROOT)


## 2. Shared helpers

The pseudo-FDT computation (`compute_pseudo_fdt`) is identical for both
pipelines. The biology-to-pixel-units conversion (`derive_size_params`
+ `resolve_cfg`) is used by Pipeline A's blob-min-area knobs.


In [ ]:
# (all functions now imported from synaptic_ssl.pseudolabels.soma_fdt)


---
# Pipeline A — SOMA-only (round blobs)

High FDT-percentile threshold + solidity filter + shape regularisation.
Output is `mask` covering only round / thick structures (cell bodies),
zero dendritic lines.


### 3.1 Soma CONFIG


In [ ]:
SOMA_CONFIG = dict(
    # ---- Data ----------------------------------------------------------
    no_patch_root="/run/media/anokhin/WinDocuments/anokhver/thesis/data/Microscopy_no_patch/20251030",
    structural_channel=2,
    file_index=1,
    batch_size=10,

    # ---- Hardware + biology --------------------------------------------
    pixel_size_nm=107.0,
    min_soma_diameter_um=3.0,
    max_hole_diameter_um=2.5,
    neck_width_um=0.3,

    # ---- Pseudo-FDT pipeline -------------------------------------------
    smooth_sigma=3.0,
    bg_sigma_k=3.0,
    t_high_fg_pct=80.0,
    fdt_sigma=1.0,

    # ---- Blob threshold on FDT field -----------------------------------
    blob_threshold_method="percentile",  # 'percentile' | 'otsu' | 'multi_otsu'
    blob_fdt_pct=90.0,
    blob_fdt_multi_classes=3,

    # ---- Cleanup -------------------------------------------------------
    blob_min_area=None,            # None -> derived (~617 px at 107 nm/px)
    hole_max_area=None,            # None -> derived
    open_radius=None,              # None -> derived
    min_solidity=0.7,
    dilate_r=5,

    # ---- Shape regularisation ------------------------------------------
    shape_mode="convex",           # 'raw' | 'closing' | 'convex' | 'ellipse' | 'circle'
    closing_radius=8,              # used iff shape_mode='closing'

    # ---- Display -------------------------------------------------------
    overlay_dim=0.6,
    fdt_clip_pct=99.0,
)

# Sanity print so you see what the pipeline will actually use.
_resolved = resolve_cfg(SOMA_CONFIG)
print(f"pixel_size = {SOMA_CONFIG['pixel_size_nm']} nm/px")
for k in ("blob_min_area", "hole_max_area", "open_radius"):
    src = "manual" if SOMA_CONFIG.get(k) is not None else "auto"
    print(f"  {k:14s} = {_resolved[k]:>5d}   ({src})")


### 3.2 Soma pipeline + visualisation


In [ ]:
# (all functions now imported from synaptic_ssl.pseudolabels.soma_fdt)


### 3.3 Soma overrides for calibration


In [ ]:
# Uncomment any line to override the SOMA_CONFIG default for this run.
SOMA_CONFIG.update(
    # file_index=0,                          # which .npy in the folder (sorted, 0-based)

    # ---- Threshold on FDT field ----------------------------------------
    # 'percentile' adapts per image (keeps top X% by count). 'otsu' is
    # mid-strict. 'multi_otsu' keeps only the brightest, biggest cores.
    # blob_threshold_method="percentile",    # 'percentile' | 'otsu' | 'multi_otsu'
    blob_fdt_pct=95.0,                     # ^ stricter (fewer/larger blobs) | v more / dimmer blobs

    # ---- Cleanup -------------------------------------------------------
    # min_solidity is area / convex_hull -- rejects branchy / star shapes.
    min_solidity=0.4,                        # ^ stricter (only convex) | v looser (keep irregular blobs)

    # ---- Shape regularisation -----------------------------------------
    # How each kept CC is drawn:
    #   'raw'     -> jagged FDT contour (no rounding)
    #   'closing' -> binary_closing(disk(closing_radius)); gentle smoothing
    #   'convex'  -> convex_hull per CC; rounds out concavities
    #   'ellipse' -> perfect ellipse from regionprops centroid + axes
    #   'circle'  -> perfect disk at centroid, radius = equivalent_diameter/2
    shape_mode="closing",                   # 'raw' | 'closing' | 'convex' | 'ellipse' | 'circle'
    # closing_radius=8,                      # px; used iff shape_mode='closing'; ^ smoother
    # pre_merge_radius=0,                    # px; > 0 bridges fragmented CCs before shape draw

    # dilate_r=5,                            # px; final dilation -- ^ thicker mask boundary
)
_r = resolve_cfg(SOMA_CONFIG)
print(
    f"file_index   = {SOMA_CONFIG['file_index']}\n"
    f"thr_method   = {SOMA_CONFIG['blob_threshold_method']}  "
    f"pct={SOMA_CONFIG['blob_fdt_pct']}  K={SOMA_CONFIG['blob_fdt_multi_classes']}\n"
    f"min_solidity = {SOMA_CONFIG['min_solidity']}\n"
    f"shape_mode   = {SOMA_CONFIG['shape_mode']}  dilate_r={SOMA_CONFIG['dilate_r']}\n"
    f"sizes (resolved): blob_min={_r['blob_min_area']} hole_max={_r['hole_max_area']} open_r={_r['open_radius']}"
)


### 3.4 Soma calibration — one image


In [ ]:
_soma_files = sorted(Path(SOMA_CONFIG["no_patch_root"]).glob("*.npy"))
if not _soma_files:
    raise FileNotFoundError(f"no .npy MIPs found under {SOMA_CONFIG['no_patch_root']}")
_soma_path = _soma_files[SOMA_CONFIG["file_index"]]
print(f"[{SOMA_CONFIG['file_index']}/{len(_soma_files)-1}] {_soma_path.name}")

_struct, _fdt, _blob = run_soma_on_image(_soma_path, SOMA_CONFIG)
print(
    f"support={_fdt['support'].mean():.3f}  "
    f"thr={_blob['threshold']:.2f}  "
    f"raw_ccs={_blob['n_raw']} -> kept={_blob['n_kept']} -> final={_blob['n_blobs']}  "
    f"cov={_blob['mask'].mean():.2%}"
)
fig, _ = visualise_soma_mask(_struct, _fdt, _blob, SOMA_CONFIG)
fig.suptitle(_soma_path.stem[:60], y=1.02, fontsize=10)
plt.show()


### 3.5 Soma batch — N images


In [ ]:
_soma_files = sorted(Path(SOMA_CONFIG["no_patch_root"]).glob("*.npy"))
if not _soma_files:
    raise FileNotFoundError(f"no .npy MIPs found under {SOMA_CONFIG['no_patch_root']}")
_soma_batch = _soma_files[4 : 4 + int(SOMA_CONFIG["batch_size"])]
print(f"soma batch: {len(_soma_batch)} images")


In [ ]:

fig, axes_grid = plt.subplots(len(_soma_batch), 3, figsize=(15, 5 * len(_soma_batch)))
if len(_soma_batch) == 1:
    axes_grid = np.atleast_2d(axes_grid)

_soma_stats = []
for i, path in enumerate(_soma_batch):
    structural, fdt_info, blob_info = run_soma_on_image(path, SOMA_CONFIG)
    visualise_soma_mask(structural, fdt_info, blob_info, SOMA_CONFIG,
                         axes=axes_grid[i], title_prefix=f"[{i}] ")
    axes_grid[i, 0].set_title(f"[{i}] {path.stem[:30]}…")
    _soma_stats.append(dict(
        idx=i, name=path.name,
        n_blobs=blob_info["n_blobs"], cov=float(blob_info["mask"].mean()),
        thr=blob_info["threshold"],
    ))
    print(f"  [{i:2d}] {path.name[:50]:50s}  "
          f"n_blobs={blob_info['n_blobs']:3d}  cov={blob_info['mask'].mean():.2%}  "
          f"thr={blob_info['threshold']:.2f}")
plt.tight_layout()
plt.show()

_n = [s["n_blobs"] for s in _soma_stats]
_c = [s["cov"] for s in _soma_stats]
print(
    f"\nSOMA batch summary ({len(_soma_stats)} images, "
    f"method={SOMA_CONFIG['blob_threshold_method']}, shape={SOMA_CONFIG['shape_mode']}):\n"
    f"  n_blobs: mean={np.mean(_n):.1f}  range=[{min(_n)}, {max(_n)}]\n"
    f"  cov:     mean={np.mean(_c):.2%}  range=[{min(_c):.2%}, {max(_c):.2%}]"
)
